# Stage 6 — Scaling up what worked

**Question:** do more TabICL estimators, a second TabICL encoding and more TabM seeds help further?

- TabICL with 32 estimators (vs. 16) on new fold splits, plus TabICL on an ordinal encoding.
- TabM with 5 new seeds averaged with stage 5's 3 seeds.
- All stage 4 and stage 5 predictions loaded from an input dataset.

| result | CV logloss | public LB | wrong rows |
|---|---|---|---|
| prob_blend | 0.27695 | 0.108963 | 1099 |
| logit_blend | 0.27711 | 0.109360 | 1103 |

**Finding:** no CV gain: the bigger TabICL mostly replaced the old one rather than adding information. The model zoo had plateaued.

**Runs on Kaggle** (GPU T4 x2, Internet on for `pip install`). Inputs and the saved-prediction chain are described in the [README](../README.md#reproducing).

In [1]:
import os, sys, glob, time, inspect, subprocess, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

def find_file(name):
    # also accepts the file saved as .txt (browsers sometimes do that when downloading CSVs)
    for n in [name, name.replace('.csv', '.txt')]:
        hits = glob.glob(f'/kaggle/input/**/{n}', recursive=True) + glob.glob(n)
        if hits:
            return os.path.abspath(hits[0])
    return None

TRAIN_PATH, TEST_PATH = find_file('Train.csv'), find_file('Test.csv')
if not TRAIN_PATH:
    raise FileNotFoundError('Train.csv not found - attach the dataset via "Add Input".')
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'

# ---- knobs ----------------------------------------------------------------
FAST = os.environ.get('FAST') == '1'            # quick smoke test
N_FOLDS = 5
RUN = dict(tabicl32=True, tabicl_ord=True, tabm=True, tabpfn=True)
BUDGET_MIN = dict(tabicl32=40, tabicl_ord=30, tabm=20, tabpfn=35)   # per model; stops adding seeds past this
SEEDS = dict(tabicl32=[2, 3, 4, 5, 6], tabicl_ord=[0, 1, 2, 3, 4],  # fold splits; v5 used 0-2 for TabICL/TabM
             tabm=[3, 4, 5, 6, 7], tabpfn=[0, 1, 2])
if FAST:
    SEEDS = {k: v[:1] for k, v in SEEDS.items()}
TABICL_ESTIMATORS = 4 if FAST else 32
TABPFN_ESTIMATORS = 2 if FAST else 4
DROP_OLD = ['realmlp']                          # v5 models left out of the blend
FULL_REFIT, FULL_WEIGHT = True, 0.5
# ---------------------------------------------------------------------------

def pip_install(*pkgs):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], capture_output=True, text=True)
    print('pip install', *pkgs, '->', 'ok' if r.returncode == 0 else 'FAILED\n' + r.stderr[-800:])
    return r.returncode == 0

for pkg in ['tabicl', 'pytabkit'] + (['tabpfn'] if RUN['tabpfn'] else []):
    pip_install(pkg)

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('train:', TRAIN_PATH, '| out:', OUT_DIR, '| device:', DEVICE, '| FAST:', FAST, '| CPUs:', os.cpu_count())

pip install tabicl -> ok
pip install pytabkit -> ok
pip install tabpfn -> ok
train: /kaggle/input/datasets/jeffmogaka/datadef/Train.csv | out: /kaggle/working | device: cuda | FAST: False | CPUs: 4


## Data and three encodings

In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
y = (train['bank_account'] == 'Yes').astype(int).values
test_ids = test['uniqueid'] + ' x ' + test['country']

CATS = ['country', 'location_type', 'cellphone_access', 'gender_of_respondent',
        'relationship_with_head', 'marital_status', 'education_level', 'job_type']
NUMS = ['household_size', 'age_of_respondent']

X = train[CATS + NUMS].reset_index(drop=True)
X_test = test[CATS + NUMS].reset_index(drop=True)
both = pd.concat([X, X_test], ignore_index=True)
split = lambda df: (df.iloc[:len(X)].reset_index(drop=True), df.iloc[len(X):].reset_index(drop=True))

# 1) one-hot + numeric (TabICL as in v5, TabPFN)
OH = pd.get_dummies(both[CATS], dtype=np.float32)
OH[NUMS] = both[NUMS].astype(np.float32)
Xoh, Xoh_test = split(OH)

# 2) ordinal: one column per category, codes ordered by frequency (no target used); education in its natural order
EDU_ORDER = ['No formal education', 'Other/Dont know/RTA', 'Primary education',
             'Secondary education', 'Vocational/Specialised training', 'Tertiary education']
ORD = pd.DataFrame(index=both.index)
for c in CATS:
    order = EDU_ORDER if c == 'education_level' and set(both[c]) <= set(EDU_ORDER) else list(both[c].value_counts().index)
    ORD[c] = both[c].map({v: i for i, v in enumerate(order)}).astype(np.float32)
ORD[NUMS] = both[NUMS].astype(np.float32)
Xord, Xord_test = split(ORD)

# 3) pandas categoricals (TabM embeds categories itself)
CAT_DF = both.copy()
for c in CATS:
    CAT_DF[c] = pd.Categorical(CAT_DF[c])
CAT_DF[NUMS] = CAT_DF[NUMS].astype(np.float32)
Xcat, Xcat_test = split(CAT_DF)

print(X.shape, '| one-hot', Xoh.shape[1], '| ordinal', Xord.shape[1], '| positive rate %.4f' % y.mean())

(23524, 10) | one-hot 39 | ordinal 10 | positive rate 0.1408


## Harness (same folds as v4/v5, time budget, failures skip the model)

In [3]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

FOLDS = {s: list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=s).split(X, y)) for s in range(10)}
ALL_ROWS = np.arange(len(y))
OOF, TEST_FOLD, TEST_FULL, SCORES, NEW_MODELS = {}, {}, {}, {}, []

def error_rate(p, t=0.5):
    return np.mean((p > t).astype(int) != y)

def run_model(name, fit, seeds, budget_min=None):
    t0, done = time.time(), []
    oof = np.zeros((len(seeds), len(y)))
    tst, full = np.zeros(len(X_test)), np.zeros(len(X_test))
    for k, s in enumerate(seeds):
        if done and budget_min and (time.time() - t0) / 60 * (len(done) + 1) / len(done) > budget_min:
            print(f'  {name}: time budget reached, stopping after {len(done)} seed(s)')
            break
        for tr_idx, va_idx in FOLDS[s]:
            predict = fit(tr_idx, s)
            oof[k, va_idx] = predict(va_idx)
            tst += predict(None) / N_FOLDS
        if FULL_REFIT:
            full += fit(ALL_ROWS, s)(None)
        done.append(k)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    n = len(done)
    oof, tst, full = oof[done], tst / n, (full / n if FULL_REFIT else tst / n)
    per_seed = [error_rate(o) for o in oof]
    OOF[name], TEST_FOLD[name], TEST_FULL[name] = oof.mean(0), tst, full
    SCORES[name] = dict(cv_error=np.mean(per_seed), logloss=np.mean([log_loss(y, o) for o in oof]),
                        logloss_avg=log_loss(y, OOF[name]), seeds=n, minutes=(time.time() - t0) / 60)
    print(f"{name:14s} logloss {SCORES[name]['logloss']:.5f} (seed-avg {SCORES[name]['logloss_avg']:.5f}) | "
          f"CV error {np.mean(per_seed):.5f} | {n} seed(s) | {SCORES[name]['minutes']:.1f} min")

def try_model(name, fit, seeds, budget_min=None):
    if not RUN.get(name, True):
        return False
    try:
        run_model(name, fit, seeds, budget_min)
        NEW_MODELS.append(name)
        return True
    except Exception as e:
        print(f'!! {name} skipped: {type(e).__name__}: {str(e)[:600]}')
        return False

def make(cls, **kw):
    # Pass only the arguments this library version accepts, so API differences don't crash the run.
    params = inspect.signature(cls.__init__).parameters
    if not any(p.kind == p.VAR_KEYWORD for p in params.values()):
        kw = {k: v for k, v in kw.items() if k in params}
    return cls(**kw)

def add_saved(name, oof, fold, full):
    OOF[name], TEST_FOLD[name], TEST_FULL[name] = oof, fold, full
    SCORES[name] = dict(cv_error=error_rate(oof), logloss=np.nan, logloss_avg=log_loss(y, oof), seeds=np.nan, minutes=0.0)

## Load v4's and v5's models (not retrained)
Requires the saved predictions of stages 4 and 5 as an input dataset. v5's TabM is kept aside so it can be averaged with v6's new TabM seeds.

In [4]:
OLD_SEEDS = {}   # model name -> number of seeds behind the saved predictions (for seed-weighted averaging)
for ver, n_seeds in [('v4', None), ('v5', {'tabm': 3, 'tabicl': 2})]:
    paths = [find_file(f) for f in [f'oof_{ver}.csv', f'test_probabilities_{ver}_fold.csv', f'test_probabilities_{ver}_full.csv']]
    if not all(paths):
        print(f'!! {ver} output not attached (Add Input -> Notebooks -> your {ver} notebook). Continuing without it.')
        continue
    o, td, tf = (pd.read_csv(p) for p in paths)
    assert (o['y'].values == y).all(), f'oof_{ver}.csv rows do not match Train.csv'
    assert (td['unique_id'].values == test_ids.values).all() and (tf['unique_id'].values == test_ids.values).all(), \
        f'{ver} test rows do not match Test.csv'
    kept = [c for c in o.columns.drop('y') if c not in DROP_OLD]
    for c in kept:
        add_saved(c, o[c].values, td[c].values, tf[c].values)
        OLD_SEEDS[c] = (n_seeds or {}).get(c)
    print(f'loaded {ver}:', kept)
if not OOF:
    raise FileNotFoundError('Neither v4 nor v5 output found - attach them via Add Input -> Notebooks.')
BASE_MODELS = list(OOF)

loaded v4: ['cb_cpu', 'cb_cpu_v2feat', 'cb_cpu_slow', 'cb_cpu_deep', 'lightgbm', 'lightgbm_xt', 'logreg_cross', 'embed_nn']
loaded v5: ['tabicl', 'tabm', 'cb_optuna']


## TabICL — 32 estimators on one-hot (new fold splits) and on ordinal encoding
v5's TabICL took 13.5 min for 2 seeds at 16 estimators; 32 estimators roughly doubles that per seed, so the budget decides how many seeds fit.

In [5]:
try:
    from tabicl import TabICLClassifier
    def tabicl_fit(Xa, Xa_test):
        def fit(tr_idx, seed):
            m = make(TabICLClassifier, n_estimators=TABICL_ESTIMATORS, random_state=seed, device=DEVICE, verbose=False)
            m.fit(Xa.iloc[tr_idx].values, y[tr_idx])
            return lambda rows: m.predict_proba(Xa_test.values if rows is None else Xa.iloc[rows].values)[:, 1]
        return fit
    try_model('tabicl32', tabicl_fit(Xoh, Xoh_test), SEEDS['tabicl32'], BUDGET_MIN['tabicl32'])
    try_model('tabicl_ord', tabicl_fit(Xord, Xord_test), SEEDS['tabicl_ord'], BUDGET_MIN['tabicl_ord'])
except ImportError as e:
    print('!! tabicl not available:', e)

tabicl-classifier-v2-20260212.ckpt: reconstructing file:   0%|          |  0.00B /  110MB            

tabicl-classifier-v2-20260212.ckpt: downloading bytes:           |  0.00B            

  tabicl32: time budget reached, stopping after 3 seed(s)
tabicl32       logloss 0.27887 (seed-avg 0.27830) | CV error 0.11105 | 3 seed(s) | 39.1 min
  tabicl_ord: time budget reached, stopping after 3 seed(s)
tabicl_ord     logloss 0.27999 (seed-avg 0.27932) | CV error 0.11241 | 3 seed(s) | 28.1 min


## TabM — 5 new seeds, then averaged with v5's 3 seeds (identical settings)

In [6]:
try:
    from pytabkit import TabM_D_Classifier
    def tabm_fit(tr_idx, seed):
        m = make(TabM_D_Classifier, random_state=seed, device=DEVICE, verbosity=0, n_cv=1)
        m.fit(Xcat.iloc[tr_idx], y[tr_idx])
        return lambda rows: m.predict_proba(Xcat_test if rows is None else Xcat.iloc[rows])[:, 1]
    if try_model('tabm_v6', tabm_fit, SEEDS['tabm'], BUDGET_MIN['tabm']) and 'tabm' in OOF:
        a, b = (OLD_SEEDS.get('tabm') or 3), SCORES['tabm_v6']['seeds']
        for D in (OOF, TEST_FOLD, TEST_FULL):
            D['tabm'] = (a * D['tabm'] + b * D.pop('tabm_v6')) / (a + b)
        SCORES.pop('tabm_v6'); NEW_MODELS[NEW_MODELS.index('tabm_v6')] = 'tabm'
        SCORES['tabm'] = dict(cv_error=error_rate(OOF['tabm']), logloss=np.nan, logloss_avg=log_loss(y, OOF['tabm']),
                              seeds=a + b, minutes=np.nan)
        print(f'tabm: v5 ({a} seeds) + v6 ({b} seeds) averaged -> seed-avg logloss {SCORES["tabm"]["logloss_avg"]:.5f}')
except ImportError as e:
    print('!! pytabkit / TabM not available:', e)

tabm_v6        logloss 0.28032 (seed-avg 0.27810) | CV error 0.11278 | 5 seed(s) | 10.5 min
tabm: v5 (3 seeds) + v6 (5 seeds) averaged -> seed-avg logloss 0.27780


## TabPFN
Needs the `TABPFN_TOKEN` secret **and** the license accepted once on ux.priorlabs.ai (Licenses tab). If it fails, the message below says what is missing.

In [7]:
token = os.environ.get('TABPFN_TOKEN')
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('TABPFN_TOKEN')
    except Exception:
        token = None
if RUN['tabpfn'] and token:
    os.environ['TABPFN_TOKEN'] = token
    try:
        from tabpfn import TabPFNClassifier
        def tabpfn_fit(tr_idx, seed):
            m = make(TabPFNClassifier, n_estimators=TABPFN_ESTIMATORS, random_state=seed, device=DEVICE,
                     ignore_pretraining_limits=True)
            m.fit(Xoh.iloc[tr_idx].values, y[tr_idx])
            return lambda rows: m.predict_proba(Xoh_test.values if rows is None else Xoh.iloc[rows].values)[:, 1]
        try_model('tabpfn', tabpfn_fit, SEEDS['tabpfn'], BUDGET_MIN['tabpfn'])
    except ImportError as e:
        print('!! tabpfn not available:', e)
else:
    print('TabPFN skipped (no TABPFN_TOKEN secret attached).')

TabPFN skipped (no TABPFN_TOKEN secret attached).


## Save v6's new predictions

In [8]:
pd.DataFrame({'y': y, **{k: OOF[k] for k in NEW_MODELS}}).to_csv(f'{OUT_DIR}/oof_v6.csv', index=False)
pd.DataFrame({'unique_id': test_ids, **{k: TEST_FOLD[k] for k in NEW_MODELS}}).to_csv(f'{OUT_DIR}/test_probabilities_v6_fold.csv', index=False)
pd.DataFrame({'unique_id': test_ids, **{k: TEST_FULL[k] for k in NEW_MODELS}}).to_csv(f'{OUT_DIR}/test_probabilities_v6_full.csv', index=False)
print('new / updated models:', NEW_MODELS)

ref = 'cb_cpu_deep' if 'cb_cpu_deep' in OOF else BASE_MODELS[0]
for k in NEW_MODELS:
    print(f'{k:12s} correlation with {ref}: {np.corrcoef(OOF[k], OOF[ref])[0, 1]:.4f}')

new / updated models: ['tabicl32', 'tabicl_ord', 'tabm']
tabicl32     correlation with cb_cpu_deep: 0.9920
tabicl_ord   correlation with cb_cpu_deep: 0.9890
tabm         correlation with cb_cpu_deep: 0.9903


## Combiners, judged on held-out logloss
Same as v5. The key line compares against **v4+v5's models as they were** (v5's blend, which scored 1094 rows).

In [9]:
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression

def logit(p):
    p = np.clip(p, 1e-5, 1 - 1e-5)
    return np.log(p / (1 - p))

class ProbBlend:                       # weighted average of probabilities, weights >= 0 summing to 1
    def fit(self, F, t):
        F = np.clip(F, 1e-6, 1 - 1e-6)
        def loss(w):
            p = np.clip(F @ w, 1e-6, 1 - 1e-6)
            return -np.mean(t * np.log(p) + (1 - t) * np.log(1 - p))
        def grad(w):
            p = np.clip(F @ w, 1e-6, 1 - 1e-6)
            return -F.T @ (t / p - (1 - t) / (1 - p)) / len(t)
        k = F.shape[1]
        r = minimize(loss, np.ones(k) / k, jac=grad, method='SLSQP', bounds=[(0, 1)] * k,
                     constraints=[dict(type='eq', fun=lambda w: w.sum() - 1, jac=lambda w: np.ones(k))],
                     options=dict(maxiter=500, ftol=1e-10))
        self.w = np.clip(r.x, 0, None) / np.clip(r.x, 0, None).sum()
        return self
    def predict(self, F):
        return F @ self.w

class LogitBlend:
    def fit(self, F, t):
        self.m = LogisticRegression(C=1.0, max_iter=5000).fit(logit(F), t); return self
    def predict(self, F):
        return self.m.predict_proba(logit(F))[:, 1]

CANDIDATES = {'prob_blend': ProbBlend, 'logit_blend': LogitBlend}
META_SEEDS = [2024, 7, 99]

def held_out(make, F):
    out = np.zeros(len(y))
    for ms in META_SEEDS:
        for tr, va in StratifiedKFold(5, shuffle=True, random_state=ms).split(F, y):
            out[va] += make().fit(F[tr], y[tr]).predict(F[va]) / len(META_SEEDS)
    return out

names = list(OOF)
P = np.column_stack([OOF[n] for n in names])
HELD_OOF = {k: held_out(make, P) for k, make in CANDIDATES.items()}
for k in CANDIDATES:
    print(f'{k:12s} held-out logloss {log_loss(y, HELD_OOF[k]):.5f} | error {error_rate(HELD_OOF[k]):.5f}')
print('v5 blend (LB 1094 rows) had held-out logloss 0.27691.  Change from v6:',
      f'{log_loss(y, HELD_OOF["prob_blend"]) - 0.27691:+.5f} (negative = better)')

T_MIX = np.column_stack([(1 - FULL_WEIGHT) * TEST_FOLD[n] + FULL_WEIGHT * TEST_FULL[n] for n in names])
FITTED = {k: make().fit(P, y) for k, make in CANDIDATES.items()}
ranked = sorted(CANDIDATES, key=lambda k: log_loss(y, HELD_OOF[k]))
print('\nranking:', ranked)
print('prob_blend weights:', {n: round(float(w), 3) for n, w in zip(names, FITTED['prob_blend'].w) if w > 0.0005})

prob_blend   held-out logloss 0.27695 | error 0.11150
logit_blend  held-out logloss 0.27711 | error 0.11142
v5 blend (LB 1094 rows) had held-out logloss 0.27691.  Change from v6: +0.00004 (negative = better)

ranking: ['prob_blend', 'logit_blend']
prob_blend weights: {'cb_cpu_deep': 0.204, 'lightgbm': 0.096, 'logreg_cross': 0.043, 'embed_nn': 0.022, 'tabicl': 0.025, 'tabm': 0.186, 'cb_optuna': 0.135, 'tabicl32': 0.289}


## Cutoff check (noise floor 0.001, as before)

In [10]:
grid = np.round(np.arange(0.40, 0.601, 0.01), 2)
THRESH = {}
for k in CANDIDATES:
    o = HELD_OOF[k]
    errs = np.array([error_rate(o, t) for t in grid])
    THRESH[k] = grid[errs.argmin()] if errs.min() < error_rate(o) - 0.001 else 0.5
    print(f'{k:12s} best cutoff {grid[errs.argmin()]} (err {errs.min():.5f}) -> using {THRESH[k]}')

prob_blend   best cutoff 0.49 (err 0.11116) -> using 0.5
logit_blend  best cutoff 0.48 (err 0.11125) -> using 0.5


## Board (sorted by logloss)

In [11]:
board = pd.DataFrame(SCORES).T[['logloss', 'logloss_avg', 'cv_error', 'seeds', 'minutes']]
board['new'] = ['NEW' if k in NEW_MODELS else '' for k in board.index]
for k in CANDIDATES:
    board.loc['>> ' + k] = [np.nan, log_loss(y, HELD_OOF[k]), error_rate(HELD_OOF[k], THRESH[k]), np.nan, np.nan, '']
display(board.sort_values('logloss_avg').round(5))
print('Reference held-out logloss: v2 0.27754 (1097 rows), v4 0.27729 (1100), v5 0.27691 (1094).')

,logloss,logloss_avg,cv_error,seeds,minutes,new
>> prob_blend,NaN,0.27695,0.11150,NaN,NaN,
>> logit_blend,NaN,0.27711,0.11142,NaN,NaN,
cb_cpu_deep,NaN,0.27764,0.11206,NaN,0.00000,
cb_optuna,NaN,0.27774,0.11201,NaN,0.00000,
tabm,NaN,0.27780,0.11257,8.0,NaN,NEW
cb_cpu_slow,NaN,0.27781,0.11172,NaN,0.00000,
cb_cpu,NaN,0.27798,0.11167,NaN,0.00000,
cb_cpu_v2feat,NaN,0.27828,0.11201,NaN,0.00000,
tabicl32,0.27887,0.27830,0.11105,3.0,39.13175,NEW
tabicl,NaN,0.27844,0.11155,NaN,0.00000,


Reference held-out logloss: v2 0.27754 (1097 rows), v4 0.27729 (1100), v5 0.27691 (1094).


## Write submissions
- `submission_v6_best.csv`: best combiner. (primary submission)
- `submission_v6_second.csv`: runner-up, as a spare.

In [12]:
def save(name, probs, t=0.5):
    sub = pd.DataFrame({'unique_id': test_ids, 'bank_account': (probs > t).astype(int)})
    path = f'{OUT_DIR}/submission_v6_{name}.csv'
    sub.to_csv(path, index=False)
    print(f'{path}: {sub.bank_account.sum()} ones / {len(sub)} rows')
    return sub

best, second = ranked[0], ranked[1]
print('best =', best, '| second =', second)
SUBS = {'best': save('best', FITTED[best].predict(T_MIX), THRESH[best]),
        'second': save('second', FITTED[second].predict(T_MIX), THRESH[second])}

def labels(df):
    return df.set_index('unique_id')['bank_account'].reindex(test_ids).values

for prev_name in ['submission_v5_best.csv', 'submission_v4_best.csv']:
    prev_path = find_file(prev_name)
    if prev_path:
        prev = labels(pd.read_csv(prev_path))
        for k, sub in SUBS.items():
            print(f'{k:7s} vs {prev_name}: {(labels(sub) != prev).sum()} rows differ')

best = prob_blend | second = logit_blend
/kaggle/working/submission_v6_best.csv: 723 ones / 10086 rows
/kaggle/working/submission_v6_second.csv: 739 ones / 10086 rows
